In [1]:
import os, re, shutil, unicodedata, warnings
from datetime import datetime
from typing import List, Tuple, Optional, Any

# -------- CPU/Threads/CUDA ----------
os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # preferuj prvú GPU
# Pri 12.1 toolchainoch sa niekedy hodí:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import torch
torch.set_num_threads(12)

import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import transformers
transformers.logging.set_verbosity_error()

from docx import Document
from striprtf.striprtf import rtf_to_text

# -------- RAG (Chroma) --------------
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ============ KONFIG ============

# 1) Mód generovania
MODE = "title"   # 'title' alebo 'keyword'

# 2) Vstupy/Výstupy
INPUT_DIR   = "./Input"
OUTPUT_DIR  = "./Output"

# 3) Tezaurus/stopwords
THESAURUS_XLSX = "./Thesaurus/SK_Local_Thesaurus.xlsx"
THESAURUS_COL  = "slovak_terms"
STOPWORDS_TXT  = "./Input/stopword_dictionary.txt"
MODELS = {
        "primary": {
        "hf_id": "Qwen/Qwen2.5-7B-Instruct",
        "quant4bit": True,
        "dtype": "fp16",
        "gen": dict(
            max_new_tokens=24,           # titulok = krátky výstup
            do_sample=True,
            temperature=0.15,            # „chladné“ dekódovanie
            top_p=0.85,
            repetition_penalty=1.1
        ),
        "batch_size": 4,
        "enable_rag": True,
        "enable_reasoning": True,       # použijeme multi-sampling + re-ranker
        "stop_newline": True
    },
    "secondary": {
        "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "quant4bit": True,
        "dtype": "fp16",
        "gen": dict(
            max_new_tokens=24,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.1
        ),
        "batch_size": 4,
        "enable_rag": True,
        "enable_reasoning": True,
        "stop_newline": True
    },
    #"primary": {
    #    "hf_id": "marcuscedricridia/Mixmix-LlaMAX3.2-1B-Merge",
    #    "quant4bit": False,
    #    "dtype": "fp16",
    #    "gen": dict(max_new_tokens=256, do_sample=True, temperature=0.6,
    #                top_p=0.95, repetition_penalty=1.6, top_k=20, min_p=0),
    #    "batch_size": 4,
    #    "enable_rag": True,
    #    "enable_reasoning": True,
    #},
    #"secondary": {
    #    "hf_id": "marcuscedricridia/8B-Nemotaur-IT",
    #    "quant4bit": True,
    #    "dtype": "fp16",
    #    "gen": dict(max_new_tokens=128, do_sample=True, temperature=0.6,
    #                top_p=0.95, repetition_penalty=1.6, top_k=20, min_p=0),
    #    "batch_size": 4,
    #    "enable_rag": True,
    #    "enable_reasoning": True,
    #},
    "fallback": {
        "hf_id": "nvidia/Llama-3.1-Nemotron-Nano-8B-v1",
        "quant4bit": True,
        "dtype": "bf16",
        "gen": dict(max_new_tokens=24, do_sample=True, temperature=0.2,
                    top_p=0.95, repetition_penalty=1.1),
        "batch_size": 4,
        "enable_rag": True,
        "enable_reasoning": True,
    },
    }
ACTIVE_MODEL = "primary"   # 'primary' | 'secondary'

# 5) RAG
RAG_PERSIST_DIR     = "./RAG_store"
RAG_COLLECTION      = "sk_legal_local"
RAG_EMB_MODEL_NAME  = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # 768-dim
RAG_TOP_K           = 5

# 6) Text / limity
MAX_TEXT_CHARS = 16000

# ============ Pomocné funkcie ============

def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return {line.strip().lower() for line in f if line.strip()}
    except FileNotFoundError:
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m: return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)
    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))
    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]
    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f"Thesaurus not found: {xlsx_path}")
    df = pd.read_excel(xlsx_path, engine="openpyxl")
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))
    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        if not t: continue
        if len(t) == 1 or t.lower() in STOPWORDS: continue
        if t not in seen:
            seen.add(t); terms.append(t)
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    if not text: return []
    hits, t_na = {}, strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0: hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0]))) ]
    return ordered[:max_terms]

# ============ DOCX/RTF split ============

_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None: buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None: buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

# ============ LLM loader ============

def _bitsandbytes_qconf():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

def load_selected_llm(active: str):
    cfg = MODELS[active]
    hf_id = cfg["hf_id"]
    qconf = _bitsandbytes_qconf() if cfg["quant4bit"] else None
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"

    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        device_map=device_map,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconf,
        torch_dtype=None if qconf is not None else torch.float16
    )

    pipe = pipeline(
        "text-generation",
        model=model, tokenizer=tok, device_map=device_map,
        return_full_text=False, **cfg["gen"]
    )
    # „Stop“ na newline vynucujeme cez prompt + krátky max_new_tokens.
    return pipe, f"{active}:{hf_id}"

LLM_PIPE, USED_MODEL = load_selected_llm(ACTIVE_MODEL)
print(f"[LLM] Loaded {USED_MODEL}")

# ============ RAG init ============

def _ensure_chroma_vectorstore() -> Optional[Chroma]:
    if not MODELS[ACTIVE_MODEL]["enable_rag"]:
        print("[RAG] Disabled.")
        return None
    os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name=RAG_EMB_MODEL_NAME)

    def _open_store():
        return Chroma(
            collection_name=RAG_COLLECTION,
            persist_directory=RAG_PERSIST_DIR,
            embedding_function=embeddings,
        )

    try:
        vs = _open_store()
        _ = vs._collection.get(limit=1)
        print(f"[RAG] Enabled (top_k={RAG_TOP_K}).")
        return vs
    except Exception as e:
        print(f"[RAG] Vectorstore init failed, recreating: {e}")
        if os.path.isdir(RAG_PERSIST_DIR):
            shutil.rmtree(RAG_PERSIST_DIR, ignore_errors=True)
            os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
        vs = _open_store()
        print("[RAG] Vectorstore ready (fresh).")
        return vs

VECTORSTORE = _ensure_chroma_vectorstore()

def rag_upsert(doc_id: str, texts: List[str], metas: List[dict]):
    if VECTORSTORE is None or not texts: return
    try:
        VECTORSTORE.add_texts(texts=texts, metadatas=metas, ids=[f"{doc_id}_{i}" for i in range(len(texts))])
    except Exception as e:
        print(f"[RAG] upsert failed: {e}")

def rag_retrieve(query: str, top_k: int = RAG_TOP_K) -> List[str]:
    if VECTORSTORE is None: return []
    try:
        docs = VECTORSTORE.similarity_search(query, k=top_k)
        return [d.page_content for d in docs]
    except Exception as e:
        print(f"[RAG] retrieve failed: {e}")
        return []

# ============ Prompting ============

def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars: return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

# Few-shot pre titulky
FEWSHOT_TITLE = [
    ("Predaj bytu a prevod vlastníctva – zmluvné strany, povinnosti, kataser",
     "Prevod vlastníctva bytu – povinné náležitosti"),
    ("Konanie o zaplatenie dlžnej sumy, uznanie dlhu, premlčanie a úroky",
     "Zaplatenie dlhu a premlčanie pohľadávky"),
    ("Zodpovednosť zamestnávateľa za škodu pri pracovnom úraze",
     "Zodpovednosť zamestnávateľa za pracovný úraz"),
]

def _fewshot_block(pairs):
    out = []
    for src, tgt in pairs:
        out.append(f"TEXT:\n{src}\nNADPIS:\n{tgt}\n---")
    return "\n".join(out)

def prompt_title(text: str, th_priority: List[str], th_sample: List[str], rag_ctx: List[str]) -> str:
    pri  = " • " + "\n • ".join(th_priority[:15]) if th_priority else "(žiadne priame zhody)"
    samp = " • " + "\n • ".join(th_sample[:25])
    rag  = ("\n\nKONTEXT:\n" + "\n---\n".join(rag_ctx[:3])) if rag_ctx else ""
    few  = _fewshot_block(FEWSHOT_TITLE[:2])  # 2 ukážky stačia

    # Trik: nadpis vždy na 1 riadok → krátky max_new_tokens + po „NADPIS:“ nechávame prázdno
    return (
        "ÚLOHA: Napíš JEDEN presný slovenský právny nadpis.\n"
        "PRAVIDLÁ: 3–10 slov • žiadna bodka • žiadne úvodzovky • jeden riadok • vecný, nie opisný.\n"
        "AK SA NEDÁ: vyber najbližší nadpis k tezauru. Výstup musí byť iba 1 riadok.\n\n"
        f"TEZAURUS – ZHODY:\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"{few}\n"
        f"TEXT:\n{text}{rag}\n\n"
        "NADPIS:"
    )

def prompt_keyword(text: str, th_priority: List[str], th_sample: List[str], rag_ctx: List[str]) -> str:
    pri  = ", ".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = ", ".join(th_sample[:40])
    rag  = ("\n\nKONTEXT:\n" + "\n---\n".join(rag_ctx[:3])) if rag_ctx else ""
    return (
        "ÚLOHA: Vyber JEDEN najvhodnejší slovenský právny pojem (1–3 slová) k nasledujúcemu textu.\n"
        "Preferuj pojmy z tezauru. Vráť len samotný pojem, bez vysvetlenia a bez bodky. Jeden riadok.\n\n"
        f"TEZAURUS – PRIO: {pri}\n"
        f"TEZAURUS – VÝBER: {samp}\n\n"
        f"TEXT:\n{text}{rag}\n\n"
        "POJEM:"
    )

# Normalizácia výstupu
def _normalize_output_item(item) -> str:
    raw = ""
    if item is None:
        raw = ""
    elif isinstance(item, dict):
        raw = item.get("generated_text") or item.get("text") or item.get("summary_text") or ""
    elif isinstance(item, list):
        if item and isinstance(item[0], dict):
            raw = item[0].get("generated_text") or item[0].get("text") or item[0].get("summary_text") or ""
        else:
            raw = str(item)
    else:
        raw = str(item)
    txt = raw or ""
    txt = re.sub(r'^[\-\•\:\s]+', '', txt).strip()
    txt = re.sub(r'[\.，。…]+$', '', txt).strip()
    txt = txt.strip(' "„”')
    lines = [ln for ln in txt.splitlines() if ln.strip()]
    return lines[0].strip() if lines else ""

def _shape_group(flat_out, n_inputs: int, nrs: int):
    blob = []
    for item in flat_out:
        if isinstance(item, list) and item and isinstance(item[0], dict):
            blob.extend(item)
        else:
            blob.append(item)
    grouped, it = [], iter(blob)
    for _ in range(n_inputs):
        chunk = []
        for _ in range(nrs):
            try:    chunk.append(next(it))
            except StopIteration: chunk.append(None)
        grouped.append(chunk)
    return grouped

# ============ Batched volanie + re-ranker ============

def safe_llm_call_batch(
    pipe, inputs, *, batch_size=1, num_return_sequences=1,
    max_new_tokens=None, temperature=None, top_p=None,
    max_retries=2, min_batch=1, min_new_tokens=16, min_samples=1
):
    import time
    bs  = max(1, int(batch_size))
    nrs = max(1, int(num_return_sequences))
    mnt = max_new_tokens

    def _call(_bs, _nrs, _mnt):
        kwargs = dict(batch_size=_bs, num_return_sequences=_nrs, return_full_text=False)
        if _mnt is not None: kwargs["max_new_tokens"] = _mnt
        if temperature is not None: kwargs["temperature"] = temperature
        if top_p is not None: kwargs["top_p"] = top_p
        return pipe(inputs, **kwargs)

    for _ in range(max_retries + 1):
        try:
            out = _call(bs, nrs, mnt)
            return out, bs, nrs, mnt
        except RuntimeError as e:
            msg = str(e)
            if ("CUDA out of memory" in msg) or ("device-side assert" in msg):
                changed = False
                if bs > min_batch: bs = max(min_batch, bs // 2); changed = True
                elif nrs > min_samples: nrs = max(min_samples, nrs // 2); changed = True
                elif mnt and mnt > min_new_tokens: mnt = max(min_new_tokens, mnt // 2); changed = True
                if not changed: break
                torch.cuda.empty_cache(); time.sleep(0.05)
            else:
                break

    # per-input fallback
    outputs, real_mnt = [], max(16, min(32, (min_new_tokens if min_new_tokens else (mnt or 24))))
    for prompt in inputs:
        try:
            out = pipe(prompt, batch_size=1, num_return_sequences=1, max_new_tokens=real_mnt, return_full_text=False)
            outputs.append(out[0] if isinstance(out, list) else out)
        except Exception:
            outputs.append("")
    return outputs, 1, 1, real_mnt

# Re-ranker: preferuje 3–10 slov, pokrytie tezauru, vyhýba sa akademickým frázam
BANNED_TOKENS = re.compile(r"\b(kapitola|úvod|diagnostika|prezentácia|štúdia|workshop|projekt|analýza)\b", re.IGNORECASE)

def _score_title(t: str, prio_terms: List[str]) -> float:
    if not t: return -1e9
    n = len(re.findall(r"\w+", t))
    if n < 3 or n > 10: return -1e6 + (10 - abs(n-6))
    cov = 0
    low = t.lower()
    for trm in prio_terms[:15]:
        if trm.lower() in low:
            cov += 1
    ban = -2.0 if BANNED_TOKENS.search(t) else 0.0
    brev = -0.006 * max(0, len(t) - 45)
    return 2.0 * cov + ban + brev

def batched_generate_texts_reasoned(jobs, batch_size: int, gen_cfg: dict) -> List[str]:
    prompts = [j[-1] for j in jobs]
    outputs = []
    nrs = 3  # tri kandidáty

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out, _, _, _ = safe_llm_call_batch(
            LLM_PIPE, batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=nrs,
            max_new_tokens=min(gen_cfg.get("max_new_tokens", 24), 32),
            temperature=max(0.1, gen_cfg.get("temperature", 0.15) * 1.1),
            top_p=gen_cfg.get("top_p", 0.85),
            max_retries=2, min_batch=1, min_new_tokens=16, min_samples=1
        )
        grouped = _shape_group(out, n_inputs=len(batch), nrs=nrs)

        for idx_in_batch, cand_list in enumerate(grouped):
            _, _, _, prio_terms, _prompt = jobs[i + idx_in_batch]
            texts = [_normalize_output_item(c) for c in cand_list if c is not None]
            texts = [re.sub(r"\s+", " ", t).strip() for t in texts if t]
            if not texts:
                outputs.append("")
                continue
            best = max(texts, key=lambda s: _score_title(s, prio_terms))
            outputs.append(best)

    while len(outputs) < len(prompts):
        outputs.append("")
    return outputs[:len(prompts)]

def batched_generate_texts(prompts: List[str], batch_size: int, gen_cfg: dict) -> List[str]:
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out, _, _, _ = safe_llm_call_batch(
            LLM_PIPE, batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=1,
            max_new_tokens=gen_cfg.get("max_new_tokens", 24),
            temperature=gen_cfg.get("temperature"),
            top_p=gen_cfg.get("top_p")
        )
        for item in out:
            outputs.append(_normalize_output_item(item))
    if len(outputs) != len(prompts):
        print(f"[WARN] outputs={len(outputs)} != inputs={len(prompts)}")
        while len(outputs) < len(prompts): outputs.append("")
        if len(outputs) > len(prompts): outputs = outputs[:len(prompts)]
    return outputs

# ============ DRIVER ============

def process_all(input_dir=INPUT_DIR):
    cfg = MODELS[ACTIVE_MODEL]
    rows, files = [], sorted(os.listdir(input_dir), key=str.lower)

    rag_active = bool(VECTORSTORE is not None)
    reasoning_active = bool(cfg["enable_reasoning"])

    # 1) Build jobs + RAG upsert
    jobs = []  # (mode, file, header, prio_terms, prompt)
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith(".docx"):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith(".rtf"):
            secs = split_rtf_by_headers(p)
        else:
            continue
        if not secs:
            print(f"No text in {f}"); continue

        # RAG upsert (iba sekcie, nie __DOCUMENT__)
        if rag_active:
            section_texts, metas = [], []
            for idx, (header, text) in enumerate(secs):
                if header == "__DOCUMENT__": continue
                if text.strip():
                    section_texts.append(text.strip())
                    metas.append({"file": f, "header": header, "idx": idx})
            if section_texts:
                rag_upsert(os.path.splitext(f)[0], section_texts, metas)

        for header, text in secs:
            if not text.strip(): continue
            short = truncate_text(text, MAX_TEXT_CHARS)
            prio_terms = terms_matched_in_text(text, max_terms=30)
            sample_terms = prio_terms if prio_terms else CANON_TERMS[:40]
            rag_ctx = rag_retrieve(short, top_k=min(3, RAG_TOP_K)) if rag_active else []

            if MODE == "title":
                prompt = prompt_title(short, prio_terms, sample_terms, rag_ctx)
                jobs.append(("title", f, header, prio_terms, prompt))
            else:
                prompt = prompt_keyword(short, prio_terms, sample_terms, rag_ctx)
                jobs.append(("keyword", f, header, prio_terms, prompt))
        print(f"Queued {f} with {len(secs)} sections.")

    if not jobs:
        print("No jobs queued.")
        return pd.DataFrame([])

    # 2) Inference
    if reasoning_active:
        gens = batched_generate_texts_reasoned(jobs, batch_size=cfg["batch_size"], gen_cfg=cfg["gen"])
    else:
        prompts = [j[-1] for j in jobs]
        gens = batched_generate_texts(prompts, batch_size=cfg["batch_size"], gen_cfg=cfg["gen"])

    # 3) Kompozícia
    for (mode, f, header, prio_terms, _), gen in zip(jobs, gens):
        if mode == "title":
            rows.append({
                "file": f, "header": header,
                "suggested_title": gen,
                "method": f"{'RAG+' if rag_active else ''}{'reasoned-' if reasoning_active else ''}llm ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })
        else:
            rows.append({
                "file": f, "header": header,
                "top_keyword": gen,
                "method": f"{'RAG+' if rag_active else ''}{'reasoned-' if reasoning_active else ''}llm ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })

    # 4) Export
    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if MODE == "title":
        df = pd.DataFrame(rows, columns=["file","header","suggested_title","method","priority_terms"])
        stub = "titles"
    else:
        df = pd.DataFrame(rows, columns=["file","header","top_keyword","method","priority_terms"])
        stub = "keywords"
    csv_path = os.path.join(OUTPUT_DIR, f"{stub}_{today}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {csv_path}")
    return df

# ------------------------------
# MAIN
# ------------------------------
if __name__ == "__main__":
    warnings.filterwarnings("ignore", message=r".*Some weights.*not of the same dtype.*")
    warnings.filterwarnings("ignore", message=r".*Tokenizer parallelism.*")
    torch.set_num_interop_threads(3)
    df = process_all(INPUT_DIR)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))

c:\Users\nyx3t\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\modeling_utils.py:6231: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.empty(byte_count // factor, dtype=torch.float16, device=device, requires_grad=False)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[LLM] Loaded primary:Qwen/Qwen2.5-7B-Instruct


C:\Users\nyx3t\AppData\Local\Temp\ipykernel_14060\2442042214.py:331: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=RAG_EMB_MODEL_NAME)
C:\Users\nyx3t\AppData\Local\Temp\ipykernel_14060\2442042214.py:334: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


[RAG] Enabled (top_k=5).
Queued 117694.docx with 8 sections.
Queued 117695.docx with 2 sections.
Queued 117696.docx with 11 sections.
Queued 117697.docx with 15 sections.
Queued 117698.docx with 8 sections.
Queued 117699.docx with 18 sections.
Queued 117700.docx with 1 sections.
Queued 117701.docx with 6 sections.
Queued 117702.docx with 12 sections.
Queued 117703.docx with 10 sections.


KeyboardInterrupt: 